# Modélisation — Détection de fraude bancaire

**Cours MLOps — notebook 3/4.** Entraîner, régler et comparer plusieurs modèles, puis choisir
le modèle final — chaque essai étant **tracé dans MLflow**.

**Modèles** : Random Forest, XGBoost, LightGBM, CatBoost, puis ensembles (vote souple, *stacking*).

**Protocole**

1. Baselines entraînées sur le train (SMOTE), évaluées sur la **validation**
2. Comparaison SMOTE / pondération des classes
3. Réglage des hyperparamètres — sans fuite de données
4. Ensembles
5. Choix du modèle et du **seuil de décision** sur la validation
6. Évaluation finale sur le **test**, une seule fois
7. Sauvegarde ; enregistrement dans le Model Registry au notebook 04

| Ancienne pratique | Problème | Ce notebook |
|---|---|---|
| Validation croisée sur le train *après* SMOTE | les fraudes synthétiques d'un pli sont fabriquées à partir de fraudes des autres plis → scores gonflés (CV = 1,0000) | réglage sur un jeu de **validation réel**, jamais rééchantillonné |
| Choix du meilleur modèle sur le test | le test sert à choisir puis à mesurer → estimation optimiste | choix sur la validation, test utilisé une seule fois |
| ROC-AUC comme seul critère | avec 0,6 % de fraudes, tous les modèles dépassent 0,99 : le critère ne départage plus | **PR-AUC** (précision moyenne) + ROC-AUC |
| Seuil fixe à 0,5 | les probabilités d'un modèle entraîné sur données SMOTE sont décalées | seuil optimisé sur la validation (F1 maximal) |
| `class_weight='balanced'` sur données déjà équilibrées | sans effet, et prête à confusion | une stratégie de déséquilibre à la fois |
| Résultats dans la console | non traçables, non comparables d'une exécution à l'autre | runs MLflow + commit Git + version DVC des données |

## 0. Configuration

In [ ]:
import json
import pickle
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score,
                             classification_report, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV, train_test_split
from xgboost import XGBClassifier

from project_setup import (ROOT, PREPROCESSING_DIR, MODELING_DIR, dvc_versions,
                           env_flag, env_int, git_commit, setup_mlflow)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Options (à définir dans .env ou dans l'environnement avant de lancer Jupyter)
QUICK_RUN = env_flag('QUICK_RUN')                              # démonstration rapide en cours
USE_MLFLOW = env_flag('USE_MLFLOW', default=True)
RUN_TUNING = env_flag('RUN_TUNING', default=True)
RUN_ENSEMBLES = env_flag('RUN_ENSEMBLES', default=not QUICK_RUN)
N_ITER = env_int('N_ITER', 4 if QUICK_RUN else 15)              # combinaisons testées par modèle
TUNING_MAX_ROWS = env_int('TUNING_MAX_ROWS', 50_000 if QUICK_RUN else 200_000)
MODELS_TO_TUNE = ['LightGBM', 'XGBoost'] if QUICK_RUN else ['RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']

MODELS_DIR = MODELING_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"QUICK_RUN={QUICK_RUN} | MLflow={USE_MLFLOW} | tuning={RUN_TUNING} (n_iter={N_ITER}, "
      f"≤{TUNING_MAX_ROWS:,} lignes) | ensembles={RUN_ENSEMBLES}")

## 1. Chargement des données préparées

Produites par le notebook 02 (`artifacts/preprocessing/preprocessed_data.pkl`, versionné avec DVC).

In [ ]:
with open(PREPROCESSING_DIR / 'preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

missing_keys = {'X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test'} - data.keys()
if missing_keys:
    raise KeyError(f"Clés absentes {missing_keys} : relancez 02_preprocessing.ipynb "
                   "(la nouvelle version produit un jeu de validation).")

X_train, y_train = data['X_train'], data['y_train']               # rééchantillonné par SMOTE
X_val, y_val = data['X_val'], data['y_val']                       # distribution réelle
X_test, y_test = data['X_test'], data['y_test']                   # distribution réelle, tenu à l'écart
X_train_orig, y_train_orig = data.get('X_train_orig'), data.get('y_train_orig')   # sans SMOTE
FEATURES = X_train.columns.tolist()

pd.DataFrame({
    'lignes': [len(y_train), len(y_val), len(y_test)],
    'fraudes': [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    'taux_fraude_%': [y_train.mean() * 100, y_val.mean() * 100, y_test.mean() * 100],
}, index=['train (SMOTE)', 'validation', 'test']).round(2)

## 2. Métriques et seuil de décision

- **PR-AUC** (`average_precision`) : aire sous la courbe précision–rappel. Une valeur de
  référence « au hasard » vaut le taux de fraude (≈ 0,006), pas 0,5 : elle reflète la difficulté
  réelle du problème et départage les modèles là où la ROC-AUC sature.
- **Seuil** : un modèle renvoie une probabilité ; la décision « fraude » s'obtient en la
  comparant à un seuil. On choisit sur la **validation** le seuil qui maximise le F1 (compromis
  précision / rappel). En contexte bancaire, on pourrait aussi pondérer par le coût d'une fraude
  manquée et celui d'une fausse alerte.

In [ ]:
def compute_metrics(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        'pr_auc': average_precision_score(y_true, proba),
        'roc_auc': roc_auc_score(y_true, proba),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
    }


def find_best_threshold(y_true, proba):
    """Seuil maximisant le F1 — à calculer sur la validation, jamais sur le test."""
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    best = int(np.argmax(f1[:-1]))   # le dernier point de la courbe n'a pas de seuil associé
    return float(thresholds[best]), float(f1[best])


def loggable_params(model):
    """Hyperparamètres simples (MLflow n'enregistre que des valeurs scalaires)."""
    return {k: v for k, v in model.get_params(deep=False).items()
            if v is None or isinstance(v, (int, float, str, bool))}

## 3. Suivi des expériences avec MLflow

Chaque modèle entraîné devient un **run** MLflow contenant :

- ses hyperparamètres et sa stratégie de déséquilibre ;
- ses métriques de validation (`val_pr_auc`, `val_roc_auc`, `val_f1`, `val_best_threshold`...) ;
- le modèle sérialisé (artefact `model/`, réutilisé par le notebook 04 et par Jenkins) ;
- le **commit Git** et le **hash DVC** des données : on sait ainsi exactement avec quel code
  et quelles données chaque modèle a été produit.

Le serveur est défini par `MLFLOW_TRACKING_URI` dans le fichier `.env` (DagsHub). Sans ce
fichier, MLflow fonctionne en local (`mlflow.db` + `mlruns/`) ; on consulte alors les runs avec
`mlflow ui --backend-store-uri sqlite:///mlflow.db` lancé depuis la racine du dépôt.

In [ ]:
RUN_TAGS = {'git_commit': git_commit() or 'inconnu', 'notebook': '03_fraud_detection_modeling', **dvc_versions()}
experiment = setup_mlflow() if USE_MLFLOW else None
print(f"Tags communs : {RUN_TAGS}")

results = {}   # clé "<modèle>_<étape>" -> modèle entraîné, probabilités et métriques de validation


def train_and_log(name, model, stage, X_fit=None, y_fit=None, imbalance='smote', extra_params=None):
    """Entraîne, évalue sur la validation, puis enregistre le run dans MLflow."""
    X_fit = X_train if X_fit is None else X_fit
    y_fit = y_train if y_fit is None else y_fit

    start = time.time()
    model.fit(X_fit, y_fit)
    train_time = time.time() - start

    proba_val = model.predict_proba(X_val)[:, 1]
    metrics = compute_metrics(y_val, proba_val)
    threshold, f1_best = find_best_threshold(y_val, proba_val)
    metrics.update(best_threshold=threshold, f1_at_best_threshold=f1_best)

    run_id = None
    if USE_MLFLOW:
        with mlflow.start_run(run_name=f"{name}_{stage}") as run:
            mlflow.set_tags({**RUN_TAGS, 'stage': stage, 'model_name': name})
            mlflow.log_params({'model_name': name, 'stage': stage, 'imbalance_strategy': imbalance,
                               'n_train_rows': len(X_fit), 'n_features': X_fit.shape[1]})
            mlflow.log_params(loggable_params(model))
            if extra_params:
                mlflow.log_params(extra_params)
            mlflow.log_metrics({f'val_{k}': v for k, v in metrics.items()})
            mlflow.log_metric('training_time_seconds', train_time)
            mlflow.sklearn.log_model(model, artifact_path='model', input_example=X_val.head(5))
            run_id = run.info.run_id

    key = f"{name}_{stage}"
    results[key] = {'name': name, 'stage': stage, 'imbalance': imbalance, 'model': model,
                    'proba_val': proba_val, 'metrics': metrics, 'train_time': train_time, 'run_id': run_id}
    print(f"{key:<28} PR-AUC={metrics['pr_auc']:.4f}  ROC-AUC={metrics['roc_auc']:.4f}  "
          f"F1@0.5={metrics['f1']:.4f}  F1@{threshold:.2f}={f1_best:.4f}  ({train_time:.1f}s)")
    return results[key]

## 4. Modèles de base (baselines)

Les données d'entraînement sont déjà équilibrées par SMOTE : on **n'ajoute pas** de
pondération des classes (`class_weight`, `scale_pos_weight`), qui serait sans effet sur des
classes 50/50 et mélangerait deux stratégies.

In [ ]:
baseline_models = {
    'RandomForest': RandomForestClassifier(n_estimators=50 if QUICK_RUN else 100, max_depth=20,
                                           min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8,
                             colsample_bytree=0.8, tree_method='hist', eval_metric='aucpr',
                             n_jobs=-1, random_state=RANDOM_STATE),
    'LightGBM': LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31,
                               n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_seed=RANDOM_STATE,
                                   verbose=False, allow_writing_files=False),
}

for name, model in baseline_models.items():
    train_and_log(name, model, 'baseline')

## 5. SMOTE ou pondération des classes ?

Alternative à SMOTE : entraîner sur les données **réelles** (non rééchantillonnées) en donnant
plus de poids aux fraudes dans la fonction de perte. Pas de données synthétiques, un
entraînement plus rapide (≈ 2× moins de lignes). On compare sur LightGBM.

In [ ]:
if X_train_orig is None:
    print("X_train_orig absent : relancez le notebook 02 pour activer cette comparaison.")
else:
    weighted = LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, class_weight='balanced',
                              n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    train_and_log('LightGBM', weighted, 'class_weight', X_fit=X_train_orig, y_fit=y_train_orig,
                  imbalance='class_weight')
    comparison_imbalance = pd.DataFrame({key: results[key]['metrics'] for key in ['LightGBM_baseline', 'LightGBM_class_weight']}).T
    display(comparison_imbalance.round(4))

**À comparer** : la PR-AUC (qualité du classement des transactions) et le **seuil optimal**.
Les deux stratégies gonflent la probabilité de fraude par rapport au taux réel (0,6 %), chacune
à sa manière : le bon seuil n'est donc pas 0,5 et diffère d'une stratégie à l'autre. C'est une
raison de plus pour toujours régler le seuil sur la validation.

## 6. Réglage des hyperparamètres — sans fuite de données

`RandomizedSearchCV` teste `N_ITER` combinaisons tirées au hasard. Au lieu d'une validation
croisée sur les données SMOTE (qui fuit), on utilise un **`PredefinedSplit`** : chaque
combinaison est entraînée sur le train (SMOTE) et notée sur la **validation réelle**, avec la
PR-AUC comme critère. Une seule évaluation par combinaison : c'est aussi beaucoup plus rapide
que 3 plis.

Pour limiter le temps de calcul, le réglage se fait sur un échantillon stratifié de
`TUNING_MAX_ROWS` lignes du train ; le modèle retenu est ensuite ré-entraîné sur tout le train.

In [ ]:
param_distributions = {
    'RandomForest': {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, 30, None],
                     'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
                     'max_features': ['sqrt', 'log2']},
    'XGBoost': {'n_estimators': [200, 400, 600], 'max_depth': [4, 6, 8, 10],
                'learning_rate': [0.03, 0.05, 0.1, 0.2], 'subsample': [0.7, 0.8, 0.9, 1.0],
                'colsample_bytree': [0.6, 0.8, 1.0], 'min_child_weight': [1, 3, 5], 'gamma': [0, 0.1, 0.3]},
    # subsample n'a d'effet dans LightGBM que si subsample_freq > 0
    'LightGBM': {'n_estimators': [200, 400, 600], 'learning_rate': [0.03, 0.05, 0.1, 0.2],
                 'num_leaves': [31, 63, 127], 'max_depth': [-1, 6, 10], 'min_child_samples': [10, 20, 50],
                 'subsample': [0.7, 0.8, 1.0], 'subsample_freq': [1], 'colsample_bytree': [0.6, 0.8, 1.0]},
    'CatBoost': {'iterations': [200, 400, 600], 'depth': [4, 6, 8], 'learning_rate': [0.03, 0.05, 0.1, 0.2],
                 'l2_leaf_reg': [1, 3, 5, 9], 'border_count': [64, 128, 254]},
}

tuned = {}
if RUN_TUNING:
    if len(X_train) > TUNING_MAX_ROWS:
        X_tune, _, y_tune, _ = train_test_split(X_train, y_train, train_size=TUNING_MAX_ROWS,
                                                stratify=y_train, random_state=RANDOM_STATE)
    else:
        X_tune, y_tune = X_train, y_train
    X_search = pd.concat([X_tune, X_val], ignore_index=True)
    y_search = pd.concat([y_tune, y_val], ignore_index=True)
    validation_split = PredefinedSplit(np.r_[np.full(len(X_tune), -1), np.zeros(len(X_val))])

    for name in MODELS_TO_TUNE:
        search = RandomizedSearchCV(clone(baseline_models[name]), param_distributions[name],
                                    n_iter=N_ITER, scoring='average_precision', cv=validation_split,
                                    refit=False, n_jobs=1, random_state=RANDOM_STATE)
        start = time.time()
        search.fit(X_search, y_search)
        print(f"\n{name} : meilleure PR-AUC de validation {search.best_score_:.4f} "
              f"({time.time() - start:.0f}s) → {search.best_params_}")
        model = clone(baseline_models[name]).set_params(**search.best_params_)
        tuned[name] = train_and_log(name, model, 'tuned', extra_params={
            'search_n_iter': N_ITER, 'search_rows': len(X_tune), 'search_scoring': 'average_precision'})
else:
    print("Réglage désactivé (RUN_TUNING=0).")

## 7. Ensembles : vote souple et *stacking*

- **Vote souple** : moyenne des probabilités des quatre modèles.
- **Stacking** : une régression logistique apprend à combiner leurs probabilités, calculées par
  validation croisée interne.

Chaque algorithme y entre avec sa meilleure version (réglée si disponible). Ces ensembles
ré-entraînent tous les modèles, plusieurs fois pour le stacking : c'est l'étape la plus longue
(désactivée avec `QUICK_RUN=1`). Un ensemble n'est retenu que s'il bat nettement le meilleur
modèle seul : sinon, sa complexité (temps d'inférence, taille, maintenance) ne se justifie pas.

In [ ]:
if RUN_ENSEMBLES:
    best_versions = [(name, clone(tuned[name]['model'] if name in tuned else baseline_models[name]))
                     for name in baseline_models]
    train_and_log('Voting_Soft', VotingClassifier(best_versions, voting='soft', n_jobs=1), 'ensemble')
    train_and_log('Stacking_LR', StackingClassifier(best_versions, cv=3, n_jobs=1,
                                                   final_estimator=LogisticRegression(max_iter=1000)), 'ensemble')
else:
    print("Ensembles désactivés (RUN_ENSEMBLES=0 ou QUICK_RUN=1).")

## 8. Comparaison sur la validation

In [ ]:
comparison = pd.DataFrame([
    {'modele': key, 'algorithme': r['name'], 'etape': r['stage'], 'desequilibre': r['imbalance'],
     **{f'val_{k}': v for k, v in r['metrics'].items()}, 'temps_s': r['train_time'], 'run_id': r['run_id']}
    for key, r in results.items()
]).sort_values('val_pr_auc', ascending=False).reset_index(drop=True)

metric_cols = ['val_pr_auc', 'val_roc_auc', 'val_f1', 'val_f1_at_best_threshold']
comparison.style.format({**{c: '{:.4f}' for c in metric_cols + ['val_precision', 'val_recall', 'val_accuracy']},
                         'val_best_threshold': '{:.3f}', 'temps_s': '{:.1f}'}) \
                .background_gradient(subset=metric_cols, cmap='RdYlGn')

In [ ]:
top_keys = comparison['modele'].head(5).tolist()
fig, axes = plt.subplots(2, 2, figsize=(18, 13))

ordered = comparison.set_index('modele').iloc[::-1]
ordered[['val_pr_auc', 'val_roc_auc']].plot(kind='barh', ax=axes[0, 0], color=['#e74c3c', '#3498db'])
axes[0, 0].set_xlim(max(0, ordered['val_pr_auc'].min() - 0.05), 1)
axes[0, 0].set_title('PR-AUC et ROC-AUC (validation)', fontweight='bold')

for key in top_keys:
    precision, recall, _ = precision_recall_curve(y_val, results[key]['proba_val'])
    axes[0, 1].plot(recall, precision, lw=2, label=f"{key} ({results[key]['metrics']['pr_auc']:.3f})")
axes[0, 1].axhline(y_val.mean(), color='black', linestyle='--', label=f'Hasard ({y_val.mean():.3f})')
axes[0, 1].set_xlabel('Rappel')
axes[0, 1].set_ylabel('Précision')
axes[0, 1].set_title('Courbes précision–rappel (top 5, validation)', fontweight='bold')
axes[0, 1].legend(loc='lower left', fontsize=9)

for key in top_keys:
    fpr, tpr, _ = roc_curve(y_val, results[key]['proba_val'])
    axes[1, 0].plot(fpr, tpr, lw=2, label=f"{key} ({results[key]['metrics']['roc_auc']:.4f})")
axes[1, 0].plot([0, 1], [0, 1], 'k--', label='Hasard (0.5)')
axes[1, 0].set_xscale('log')
axes[1, 0].set_xlabel('Taux de faux positifs (échelle log)')
axes[1, 0].set_ylabel('Taux de vrais positifs')
axes[1, 0].set_title('Courbes ROC (top 5, validation)', fontweight='bold')
axes[1, 0].legend(loc='lower right', fontsize=9)

ordered['temps_s'].plot(kind='barh', ax=axes[1, 1], color='#95a5a6')
axes[1, 1].set_title("Temps d'entraînement (s)", fontweight='bold')

plt.tight_layout()
plt.show()

**Lecture.** Les ROC-AUC sont toutes proches de 1 et peu discriminantes ; l'échelle log des
faux positifs montre pourtant que les modèles diffèrent là où cela compte (très peu de
fausses alertes). La PR-AUC rend ces écarts visibles.

## 9. Choix du modèle final et du seuil

Critère : **PR-AUC de validation**. Le seuil retenu est celui qui maximise le F1 sur la
validation pour ce modèle.

In [ ]:
best_key = comparison.loc[0, 'modele']
best = results[best_key]
best_threshold = best['metrics']['best_threshold']

print(f"🏆 Modèle retenu : {best_key}")
print(f"   PR-AUC validation  : {best['metrics']['pr_auc']:.4f}")
print(f"   ROC-AUC validation : {best['metrics']['roc_auc']:.4f}")
print(f"   Seuil retenu       : {best_threshold:.3f} (F1 validation {best['metrics']['f1_at_best_threshold']:.4f})")
print(f"   Run MLflow         : {best['run_id']}")

## 10. Évaluation finale sur le test

Le test n'a servi à **aucune** décision jusqu'ici : la mesure ci-dessous est une estimation
honnête de la performance en production (sur des données de la même période).

In [ ]:
proba_test = best['model'].predict_proba(X_test)[:, 1]
test_default = compute_metrics(y_test, proba_test, threshold=0.5)
test_tuned = compute_metrics(y_test, proba_test, threshold=best_threshold)

display(pd.DataFrame({'seuil 0.5': test_default, f'seuil {best_threshold:.3f}': test_tuned}).round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, threshold in zip(axes, [0.5, best_threshold]):
    ConfusionMatrixDisplay.from_predictions(y_test, (proba_test >= threshold).astype(int),
                                            display_labels=['Légitime', 'Fraude'],
                                            cmap='Blues', values_format=',d', ax=ax, colorbar=False)
    ax.set_title(f'Test — seuil {threshold:.3f}', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, (proba_test >= best_threshold).astype(int),
                            target_names=['Légitime', 'Fraude'], digits=4))

if USE_MLFLOW and best['run_id']:
    with mlflow.start_run(run_id=best['run_id']):
        mlflow.log_metrics({f'test_{k}': v for k, v in test_tuned.items()})
        mlflow.log_metrics({f'test_{k}_threshold_0_5': v for k, v in test_default.items()})
        mlflow.log_figure(fig, 'confusion_matrix_test.png')
        mlflow.set_tag('selected', 'true')
    print(f"Métriques de test et matrice de confusion ajoutées au run {best['run_id']}.")

## 11. Interprétabilité : importance des variables

In [ ]:
final_model = best['model']
if hasattr(final_model, 'feature_importances_'):
    importances = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(10, 7))
    (importances / importances.sum()).tail(15).plot(kind='barh', ax=ax, color='#3498db')
    ax.set_title(f'Importance des variables — {best_key}', fontweight='bold')
    ax.set_xlabel("Part de l'importance totale")
    plt.tight_layout()
    plt.show()
    if USE_MLFLOW and best['run_id']:
        with mlflow.start_run(run_id=best['run_id']):
            mlflow.log_figure(fig, 'feature_importance.png')
else:
    print(f"{best_key} n'expose pas d'importance des variables (ensemble).")

Comparer ce classement à l'EDA (notebook 01) : `amt`, l'heure et la catégorie doivent dominer ;
une variable jugée non informative qui arriverait en tête mériterait une enquête (fuite de
données, erreur de preprocessing).

## 12. Sauvegarde

- `artifacts/modeling/models/best_model_final.pkl` : le modèle retenu ;
- `artifacts/modeling/models/best_model_final_metadata.json` : seuil, métriques, run MLflow,
  versions Git et DVC ;
- `artifacts/modeling/model_comparison.csv` : le tableau de comparaison.

Tous les autres modèles restent disponibles dans MLflow : inutile de les sérialiser à la main.

In [ ]:
with open(MODELS_DIR / 'best_model_final.pkl', 'wb') as f:
    pickle.dump(final_model, f)

metadata = {
    'model_key': best_key,
    'algorithm': best['name'],
    'stage': best['stage'],
    'imbalance_strategy': best['imbalance'],
    'threshold': best_threshold,
    'selection_metric': 'val_pr_auc',
    'validation_metrics': best['metrics'],
    'test_metrics': test_tuned,
    'test_metrics_threshold_0_5': test_default,
    'features': FEATURES,
    'mlflow_experiment': experiment.name if experiment else None,
    'mlflow_run_id': best['run_id'],
    'created_at': datetime.now().isoformat(timespec='seconds'),
    **RUN_TAGS,
}
with open(MODELS_DIR / 'best_model_final_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=float)

comparison.drop(columns=['run_id']).to_csv(MODELING_DIR / 'model_comparison.csv', index=False)
print(f"✓ {(MODELS_DIR / 'best_model_final.pkl').relative_to(ROOT)}")
print(f"✓ {(MODELS_DIR / 'best_model_final_metadata.json').relative_to(ROOT)}")
print(f"✓ {(MODELING_DIR / 'model_comparison.csv').relative_to(ROOT)}")

## 13. Résumé

- Tous les modèles ont été comparés sur la **même validation**, avec un critère adapté au
  déséquilibre (PR-AUC) ; le test n'a servi qu'à la mesure finale.
- Le seuil de décision fait partie du modèle : il est enregistré dans les métadonnées et dans
  MLflow. L'API utilise aujourd'hui `model.predict()` (seuil 0,5) ; appliquer le seuil retenu
  est une évolution à prévoir côté backend.
- Chaque run MLflow est relié à un commit Git et à une version DVC des données.

**Étape suivante** : `04_mlflow_tracking.ipynb` — enregistrer le modèle retenu dans le
**Model Registry**, le promouvoir et l'exporter pour l'API.